# Figure: reverse-mode autodiff overhead

Loads `results/differentiability/autodiff_overhead.json` (produced by `bench/differentiability/autodiff_overhead.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# Resolve the results directory whether the notebook is run from its own
# folder or from the repo root.
_here = Path.cwd()
_candidates = [
    _here / "results" / "differentiability",
    _here.parents[1] / "results" / "differentiability",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
PALETTE = {"radix": "#4C72B0", "octree": "#DD8452", "kdtree": "#55A868"}
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "autodiff_overhead.json").read_text())
meta = data["metadata"]
records = data["records"]
ns = [r["n"] for r in records]
fwd = [r["forward"]["min_s"] * 1e3 for r in records]
bwd = [r["forward_backward"]["min_s"] * 1e3 for r in records]
ratio = [r["overhead_ratio"] for r in records]
print(f"device={meta['device_kind']} jax={meta['jax_version']}")


In [ ]:
fig, (ax_t, ax_r) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

ax_t.plot(ns, fwd, "o-", color=PALETTE["kdtree"], label="forward")
ax_t.plot(ns, bwd, "s-", color=PALETTE["radix"], label="forward + backward")
ax_t.set_xscale("log"); ax_t.set_yscale("log")
ax_t.set_xlabel("N particles"); ax_t.set_ylabel("time [ms]")
ax_t.set_title("KD-tree pipeline: build + kNN + reduce"); ax_t.legend()

ax_r.plot(ns, ratio, "o-", color="#000000")
ax_r.axhline(1.0, color=BASELINE, ls=":", lw=1)
ax_r.set_xscale("log")
ax_r.set_xlabel("N particles"); ax_r.set_ylabel("(fwd+bwd) / fwd")
ax_r.set_title("Autodiff overhead ratio")
ax_r.set_ylim(bottom=0)

fig.suptitle(f"Reverse-mode overhead on {meta['device_kind']}", fontsize=12)
out = FIGDIR / "fig_autodiff_overhead.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
